# Keep a small dataset

This notebook exports an RDF subset with Phenobarbital, its linked pathways, and their key events. The resulting RDF document adheres to the AOP Wiki RDF schema.

## Open AOPWiki

Use the repository environment. After updating rdfsolve, restart the kernel and run from the top.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/rdfsolve").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from rdfsolve.client import Client

FILES = ROOT / "notebooks/pydantic_clients"
data = Client.open(ROOT / "notebooks/data/aopwikirdf.schema.json")

## Find Phenobarbital and follow its links

In [ ]:
chemical = data.find("Phenobarbital").of_type("Chemical entity")
stressors = chemical.related("Stressor", incoming=True)
pathways = stressors.related("Adverse outcome pathway", incoming=True)
events = pathways.related("Key event", via="has key event")

pathways

## Check the observed route

These paths start from every chemical in the selection.

In [ ]:
routes = chemical.paths_between("Adverse outcome pathway", via=["Stressor"], max_hops=2)
routes

In [ ]:
pairs = data.retrieve(routes.iloc[0]["Reference"], source="chemical", target="pathway")
pairs.table()

## Save this selection

The file keeps the selected records and the links we followed between them.

In [ ]:
data.save(FILES / "phenobarbital-subset.ttl", chemical, stressors, pathways, events)
data.save_session(FILES / "phenobarbital-session.json")
data.close()

## Search the saved file

This time, the data comes from our file.

In [ ]:
local = Client.from_session(
    FILES / "phenobarbital-session.json", data_file=FILES / "phenobarbital-subset.ttl"
)
local.find("Phenobarbital")

In [ ]:
local.close()